# 26 · Monitoring, fairness, and explainability

**Prerequisites:** 00 — Setup; basic Python. No other notebook state is required.

**Execution:** portable. Run cells from top to bottom. Each notebook is independent.

## What you will learn

Measure distribution shift, permutation importance, subgroup metrics, and monitoring limitations.


## Step 1 — Train a reference model

Feature perturbation and subgroup checks use held-out examples. Synthetic groups here are arbitrary and do not represent demographic fairness conclusions.


In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
X, y = make_classification(n_samples=1000, n_features=10, n_informative=5,
                           weights=[0.75, 0.25], random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42)

model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=500)).fit(X_train,y_train)
prob=model.predict_proba(X_test)[:,1]
print('Reference AUC:',roc_auc_score(y_test,prob))


**Check your result:** A reference metric is available for comparison.


## Step 2 — Estimate feature importance

Permutation importance measures score deterioration after shuffling a feature. Correlated features can mask one another; this is not causal attribution.


In [ ]:
from sklearn.inspection import permutation_importance
importance=permutation_importance(model,X_test,y_test,scoring='roc_auc',n_repeats=4,random_state=42,n_jobs=1)
rank=np.argsort(-importance.importances_mean)
print('Top features:',rank[:3],importance.importances_mean[rank[:3]])


**Check your result:** Features are ranked by held-out score sensitivity.


## Step 3 — Measure input shift

Standardized mean shift is a simple indicator. A drift alert does not establish that prediction quality has degraded.


In [ ]:
shifted=X_test.copy()
shifted[:,0]+=2*X_train[:,0].std()
shift=np.abs((shifted.mean(axis=0)-X_train.mean(axis=0))/(X_train.std(axis=0)+1e-9))
print('Mean shift by feature:',np.round(shift,2))
assert shift[0]>1
print('Drifted AUC (labels available in this simulation):',roc_auc_score(y_test,model.predict_proba(shifted)[:,1]))


**Check your result:** Feature zero triggers a large standardized shift.


## Step 4 — Audit subgroups

Report sample sizes and uncertainty. Small subgroup differences can be noise; a single metric cannot certify fairness.


In [ ]:
from sklearn.metrics import recall_score
group=np.arange(len(y_test))%2
for g in [0,1]:
    mask=group==g
    print({'group':g,'n':int(mask.sum()),'positive_n':int(y_test[mask].sum()),
           'recall':recall_score(y_test[mask],prob[mask]>=0.5)})


**Check your result:** Each arbitrary group has its own count and recall.


## Practice

Design a delayed-label monitoring table with model version, prediction timestamp, observed outcome and outcome timestamp.

Predict the outcome before changing the code. Explain the result in your own words.


## Solution guide

Join outcomes by a stable prediction/event key. Evaluate only matured labels to avoid biased missing-outcome samples, and separate input drift from labeled performance degradation.

## Before using this at work

Use representative data, measure quality on held-out examples, record dependencies and permissions, and review failures. Tiny synthetic results demonstrate mechanics; they do not establish production quality.
